In [44]:
from sklearn.linear_model import LinearRegression
from bokeh.models import HoverTool 
from bokeh.io import export_svg
import numpy as np
import transforms3d as td
import pandas as pd
import bokeh.io
import bokeh.plotting
# Import libraries
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import re
import panel as pn
from bokeh.models import ColumnDataSource
from bokeh.palettes import Viridis
bokeh.io.output_notebook()
pn.extension()
import plotly.graph_objects as go
from bokeh.io import export_png
import holoviews as hv
from bokeh.models import CategoricalColorMapper, Legend

Loading BokehJS ...

In [80]:
p1_opens = {'passive_start' : [[66, 110], [199, 236], [305, 348], [425, 462], [535, 578], [658, 702], [769, 810], [875, 920], [994, 1030], [1090, 1135], [1204, 1249]],
'active_rorcr' : [[1856, 1934], [2505, 2575], [3244, 3305]],
'active_cube' : [[626, 680], [1003, 1053], [1946, 1990], [3327, 3371], [4834, 4878], [5978, 6028], [6178, 6219], [6549, 6590], [6853, 6892], [7260, 7303], [7550, 7590], [9115, 9161], [9322, 9366], [9788, 9831], [10697, 10746], [11063, 11108], [16324, 16369], [17175, 17220], [17741, 17782], [18209, 18249], [18661, 18699], [23887, 23910], [24319, 24342], [24440, 24504], [26049, 26097], [26274, 26304]],
'passive_end' : [[225, 266], [319, 351], [414, 455], [501, 541], [593, 632], [705, 743], [805, 844], [896, 936], [995, 1044], [1099, 1139]]}

p13_opens = {'passive_start' : [[334, 383], [555, 600], [737, 787], [897, 933], [1031, 1075], [1166, 1211], [1303, 1343], [1437, 1477], [1558, 1598], [1683, 1730]],
             'active_rorcr1' : [[2694,2941],[3872,3921],[4662,4956],[6828,6895]],
             'active_cube' : [[3483, 3534], [3651, 3729], [4757, 4816], [4871, 4893], [5470, 5498], [6234, 6291], [6669, 6732], [7009, 7053], [7457, 7505], [8018, 8049], [8190, 8233], [8458, 8503], [8668, 8740], [9261, 9296], [9351, 9419], [9584, 9626], [10539, 10570], [10613, 10649], [10770, 10822], [11033, 11072], [11383, 11423], [11692, 11732], [12106, 12144], [12716, 12766], [12945, 12990], [13125, 13157], [13230, 13269], [13873, 13918], [14012, 14061], [15229, 15274]],
             'passive_end' : [[129,164],[265,356],[438,518],[665,701],[843,881],[963,1050],[1186,1225],[1334,1372],[1506,1536],[1647,1687]]}

p3_opens = {'passive_start' : [[316, 371], [810, 866], [1029, 1080], [1228, 1282], [1427, 1476], [1623, 1679], [1805, 1864], [1979, 2030], [2119, 2178], [2272, 2329]],
            'active_rorcr' : [[6454, 6523], [7650, 7753], [9392, 9444]],
            'active_cube' : [[2328, 2696], [4601, 4651], [5807, 5861], [14749, 14812], [15108, 15186]],
            'passive_end' : [[146, 198], [552, 610], [858, 913], [1099, 1148], [1273, 1327], [1486, 1537], [1681, 1736], [1885, 1936], [2055, 2111], [2253, 2303]]}

opens = {'p1' : p1_opens, 'p13': p13_opens, 'p3' : p3_opens}

plot_label = {'passive_start': 'P1 Phase',
 'passive_end': 'P2 Phase',
 'active_rorcr': 'T4 Phase',
 'active_rorcr1': 'T4 Phase',
 'active_cube': 'Functional Training Phase',
 'mcp_angle': 'MCP Angle',
 'pip_angle': 'PIP Angle',
 'm': 'Stiffness (N/mm)',
 'motor_position': 'Cable Retraction (mm)',
 'futek': 'Force (N)',
 'time_open': 'Time over Experimental Condition (unscaled)',
 'p1': 'S1',
 'p13': 'S2',
 'p3': 'S3'}

In [81]:
def get_df(file):
    df = pd.read_csv(file)
    df['time_unitless'] = df['index']
    df.set_index(['condition', 'c_index', 'index'], inplace=True)
    df['open'] = 0
    df['m'] = 0
    df['b'] = 0
    df['time_open']=0

    return df

def get_opens(df, opens_dict):
    d = 1
    for c in opens_dict.keys():
        o=1
        for open in opens_dict[c]:
            i = open[0]
            f = open[1]
            df.loc[(c, np.r_[i:f], slice(None)), 'open'] = o
            x_vals = df.loc[(c, np.r_[i:f], slice(None)), 'motor_position'].copy().values.reshape(-1, 1)
            y_vals = df.loc[(c, np.r_[i:f], slice(None)), 'futek'].copy().values
            model = LinearRegression()
            model.fit(x_vals, y_vals)
            df.loc[(c, np.r_[i:f], slice(None)), 'm'] = np.ones(f-i)*model.coef_
            df.loc[(c, np.r_[i:f], slice(None)), 'b'] = np.ones(f-i)*model.intercept_
            df.loc[(c, np.r_[i:f], slice(None)), 'time_open'] = d
            o += 1
            d+=1
    return df

def get_mcp_pip_plot(df, condition, plot_label):
        x = 'c_index'
        y = 'mcp_angle'
        z = 'pip_angle'
        df = df.loc[(condition, slice(None), slice(None)), :].copy().reset_index()
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                x_axis_label="Time over Experimental Condition (unscaled)",
                y_axis_label="Angle (degres)",
                title = plot_label[condition] + ' MCP and PIP angles',
                y_range = [-45, 200],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )

        p.circle(df[x], df[y], legend_label = plot_label[y], alpha = 0.5)
        p.circle(df[x], df[z], color = 'red', legend_label = plot_label[z], alpha = 0.5)

        return p

def get_angle_plots(df, conditions, plot_label):
    plts = []
    for i in conditions:
        p = get_mcp_pip_plot(df, i, plot_label)
        p.title.text_font_size = '20pt'
        p.legend.label_text_font_size = '20pt'
        p.xaxis.axis_label_text_font_size = '20pt'
        p.yaxis.axis_label_text_font_size = '20pt'
        p.yaxis.major_label_text_font_size = '18pt'
        p.xaxis.major_label_text_font_size = '18pt'
        #p.legend.click_policy = 'hide'
        bokeh.io.show(p)
        plts.append(p)
    return plts


In [82]:
patient = 'p13'
file = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/parsed_data/%s_parsed_data.csv'%patient
conditions = opens[patient].keys()
df = get_df(file)
df = get_opens(df, opens[patient])
df

raw_time                    time  \
condition                 c_index index                                         
passive_start_calibration 0       0      1.706714e+09  2024-01-31 15:08:17.72   
                          1       1      1.706714e+09  2024-01-31 15:08:17.76   
                          2       2      1.706714e+09  2024-01-31 15:08:17.78   
                          3       3      1.706714e+09  2024-01-31 15:08:17.82   
                          4       4      1.706714e+09  2024-01-31 15:08:17.84   
...                                               ...                     ...   
passive_end               1807    35550  1.706716e+09  2024-01-31 15:48:29.96   
                          1808    35551  1.706716e+09  2024-01-31 15:48:30.03   
                          1809    35552  1.706716e+09  2024-01-31 15:48:30.07   
                          1810    35553  1.706716e+09  2024-01-31 15:48:30.09   
                          1811    35554  1.706716e+09  2024-01-31 15:48:30.13   

                                         futek  motor_position  \
condition                 c_index index                          
passive_start_calibration 0       0        0.0           0.000   
                          1       1        0.0           0.000   
                          2       2        0.0           0.000   
                          3       3        0.0           0.000   
                          4       4        0.0           0.000   
...                                        ...             ...   
passive_end               1807    35550    0.0          -0.042   
                          1808    35551    0.0          -0.042   
                          1809    35552    0.0          -0.042   
                          1810    35553    0.0          -0.042   
                          1811    35554    0.0          -0.042   

                                                                               trakstar_02  \
condition                 c_index index                                                      
passive_start_calibration 0       0      [[ 0.23543548  0.97106643  0.04000161  0.07983...   
                          1       1      [[ 0.23805717  0.97042542  0.04004115  0.07978...   
                          2       2      [[ 0.23913704  0.97014569  0.04038337  0.07978...   
                          3       3      [[ 0.24041984  0.96976834  0.04180515  0.07978...   
                          4       4      [[ 0.241304    0.96949669  0.04299479  0.07978...   
...                                                                                    ...   
passive_end               1807    35550  [[-0.96637079  0.23173928 -0.11146478  0.03854...   
                          1808    35551  [[-0.96640949  0.23150155 -0.11162312  0.03856...   
                          1809    35552  [[-0.96647064  0.23136262 -0.11138153  0.03857...   
                          1810    35553  [[-0.96649038  0.23115595 -0.111639    0.03857...   
                          1811    35554  [[-0.96649038  0.23115595 -0.111639    0.03848...   

                                             mcp_angle      pip_angle  \
condition                 c_index index                                 
passive_start_calibration 0       0      1.551312e-315   0.000000e+00   
                          1       1      6.907205e-310  6.907205e-310   
                          2       2      6.907205e-310  6.907205e-310   
                          3       3      6.907205e-310  6.907205e-310   
                          4       4      6.907205e-310  6.907205e-310   
...                                                ...            ...   
passive_end               1807    35550   2.648452e+01   5.682314e+01   
                          1808    35551   2.649319e+01   5.682412e+01   
                          1809    35552   2.649653e+01   5.683374e+01   
                          1810    35553   2.649582e+01   5.684136e+01   
                          1811   

In [ ]:
plts = get_angle_plots(df, conditions, plot_label)

In [86]:
plts = []
x = 'motor_position'
y = 'futek'

hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s Force vs. Cable Retraction during Hand Opening'%plot_label[patient],
        #y_range = [-2, 20],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df1 = df[df['open'] > 0].copy()
df1.reset_index(inplace = True, drop = False)
df1.set_index(['condition', 'open'], inplace= True)
p.add_layout(Legend(), 'right')

for i, c in enumerate(conditions):
  num_opens = df1.loc[(c, slice(None))].index.values.max()

  for o in range(1, num_opens+1):
    df2 = df1.loc[(c, o)].copy()
    p.line(df2[x], df2[y], color = col[i], legend_label = c, line_width = 3, alpha = 0.5)

p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='bottom'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

/tmp/ipykernel_438558/667491960.py:27: PerformanceWarning: indexing past lexsort depth may impact performance.
  df2 = df1.loc[(c, o)].copy()


In [87]:
bokeh.palettes.Viridis256[2]

'#440357'

In [88]:
x = 'motor_position'
y = 'futek'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.Turbo256[::-1]

df1 = df[df['open'] > 0].copy()
df1.reset_index(inplace = True, drop = False)
df1.set_index(['condition', 'open'], inplace= True)

for i, c in enumerate(conditions):
  num_opens = df1.loc[(c, slice(None))].index.values.max()
  p = bokeh.plotting.figure(
    width=1000,
    height=600,
    y_axis_label=plot_label[y],
    x_axis_label=plot_label[x],
    title = '%s Force vs. Cable Retraction during Hand Opening'%plot_label[patient],
    #y_range = [-2, 20],
    tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
  p.add_layout(Legend(), 'right')
  for o in range(1, num_opens+1):
    df2 = df1.loc[(c, o)].copy()
    p.line(df2[x], df2[y], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)

  p.title.text_font_size = '20pt'
  p.legend.label_text_font_size = '8pt'
  p.xaxis.axis_label_text_font_size = '20pt'  
  p.yaxis.axis_label_text_font_size = '20pt'
  p.yaxis.major_label_text_font_size = '18pt'
  p.xaxis.major_label_text_font_size = '18pt'
  p.legend.location='top_left'
  p.legend.click_policy='hide'
  legend = bokeh.models.Legend(location=(10,10))
  p.add_layout(legend, 'right')
  #plts.append(p)
  bokeh.io.show(p)

/tmp/ipykernel_438558/2869189175.py:25: PerformanceWarning: indexing past lexsort depth may impact performance.
  df2 = df1.loc[(c, o)].copy()


In [95]:
x = 'motor_position'
y = 'futek'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.Turbo256[::-1]

df1 = df[df['open'] > 0].copy()
df1.reset_index(inplace = True, drop = False)
df1.set_index(['condition', 'open'], inplace= True)

for i, c in enumerate(conditions):
  num_opens = df1.loc[(c, slice(None))].index.values.max()
  p = bokeh.plotting.figure(
    width=1000,
    height=600,
    y_axis_label=plot_label[y],
    x_axis_label=plot_label[x],
    title = '%s Best Fit Lines during Hand Opening'%plot_label[patient],
    #y_range = [-2, 20],
    tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
  p.add_layout(Legend(), 'right')
  for o in range(1, num_opens+1):
    df2 = df1.loc[(c, o)].copy()
    t = np.linspace(0, 50, 100)
    p.line(t, df2['m'][0]*t + df2['b'][0], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)

  p.title.text_font_size = '20pt'
  p.legend.label_text_font_size = '8pt'
  p.xaxis.axis_label_text_font_size = '20pt'  
  p.yaxis.axis_label_text_font_size = '20pt'
  p.yaxis.major_label_text_font_size = '18pt'
  p.xaxis.major_label_text_font_size = '18pt'
  p.legend.location='top_left'
  p.legend.click_policy='hide'
  legend = bokeh.models.Legend(location=(10,10))
  p.add_layout(legend, 'right')
  #plts.append(p)
  bokeh.io.show(p)

/tmp/ipykernel_438558/3539126520.py:25: PerformanceWarning: indexing past lexsort depth may impact performance.
  df2 = df1.loc[(c, o)].copy()


In [89]:
x = 'motor_position'
y = 'futek'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s Force vs. Cable Retraction during Hand Opening'%plot_label[patient],
        #y_range = [-2, 20],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
p.add_layout(Legend(), 'right')
df1 = df[df['open']>0].copy()
for i, c in enumerate(conditions):
  dfd = df1.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = plot_label[c], size = 10, alpha = 0.5)
  #p.line(dfd[x], dfd[y], color = col[i], legend_label = plot_label[c])
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'

p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [90]:
x = 'time_open'
y = 'm'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s Stiffness for Hand Openings over Experiment Condition'%plot_label[patient],
        y_range = [-0.02, 0.7],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df1 = df[df['open']>0].copy()
p.add_layout(Legend(), 'right')
for i, c in enumerate(conditions):
  dfd = df1.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = plot_label[c], size = 10)
  p.segment(x0=dfd[x], x1=dfd[x], y0=0, y1=dfd[y], color = col[i], legend_label = plot_label[c])
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'

p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [91]:
x = 'time_open'
y = 'mcp_angle'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s MCP Angles for Hand Opening over Experimental Condition'%plot_label[patient],
        y_range = [-45, 125],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df1 = df[df['open']>0].copy()
p.add_layout(Legend(), 'right')
for i, c in enumerate(conditions):
  dfd = df1.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = plot_label[c])
 
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='top_left'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [92]:
x = 'time_open'
y = 'pip_angle'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s PIP Angles for Hand Opening over Experimental Condition'%plot_label[patient],
        y_range = [-45, 125],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df1 = df[df['open']>0].copy()
p.add_layout(Legend(), 'right')
for i, c in enumerate(conditions):
  dfd = df1.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = plot_label[c])
 
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='bottom_right'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [ ]:
i = 0 

for p in plts:
    export_png(p, filename='/home/jxu/hand_orthosis_ws/src/trakstar_ros/parsed_data/figures/%s_%s.png'%(plot_label[patient], i))
    i+=1

In [57]:
x = 'motor_position'
y = 'futek'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]

df1 = df[df['open']>0].copy()
for i, c in enumerate(['active_cube']):
  p = bokeh.plotting.figure(
        width=400,
        height=200,
        y_axis_label=y,
        x_axis_label=x,
        title = 'P13 Force vs. Tendon Displacement',
        #y_range = [-2, 20],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
  dfd = df1.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = c)
  p.line(dfd[x], dfd[y], color = col[i], legend_label = c)
  p.legend.location='top_left'
  p.legend.click_policy='hide'
  bokeh.io.show(p)

